# Price Explorer
Browse and visualise eBay.de sold-price data stored in `data/prices.db`.

In [1]:
import sys
import sqlite3
import pandas as pd
from pathlib import Path

# Make the bot package importable from the notebook
sys.path.insert(0, str(Path().resolve()))

DB_PATH = Path("data/prices.db")

In [2]:
# Connect to the SQLite database
con = sqlite3.connect(DB_PATH)
print(f"Connected to {DB_PATH}  ({DB_PATH.stat().st_size / 1024:.1f} KB)")

Connected to data\prices.db  (1444.0 KB)


## Latest price summary — all models

In [3]:
df_summary = pd.read_sql("""
    SELECT
        pm.brand,
        pm.model,
        ps.computed_date  AS date,
        ps.avg_price      AS avg,
        ps.median_price   AS median,
        ps.min_price      AS min,
        ps.max_price      AS max,
        ps.sample_count   AS n
    FROM price_summaries ps
    JOIN phone_models pm ON pm.id = ps.phone_model_id
    WHERE ps.computed_date = (
        SELECT MAX(ps2.computed_date)
        FROM price_summaries ps2
        WHERE ps2.phone_model_id = ps.phone_model_id
    )
    ORDER BY pm.brand, pm.model
""", con)

df_summary

,brand,model,date,avg,median,min,max,n
0,Apple,iPhone 12,2026-04-04,128.62,129.51,70.00,229.00,48
1,Apple,iPhone 12 Mini,2026-04-04,120.53,114.61,75.00,199.99,43
2,Apple,iPhone 12 Pro,2026-04-04,170.66,169.00,106.19,294.95,35
3,Apple,iPhone 12 Pro Max,2026-04-04,222.75,223.00,101.00,329.00,13
4,Apple,iPhone 13,2026-04-04,186.84,185.00,99.00,269.00,81
5,Apple,iPhone 13 Mini,2026-04-04,190.08,184.19,125.00,300.00,38
6,Apple,iPhone 13 Pro,2026-04-04,253.47,254.80,100.00,380.00,47
7,Apple,iPhone 13 Pro Max,2026-04-04,303.78,299.99,220.00,395.00,33
8,Apple,iPhone 14,2026-04-04,240.48,241.00,102.00,350.00,47
9,Apple,iPhone 14 Plus,2026-04-04,265.45,257.50,181.00,421.00,14


## Filter by brand

In [4]:
# Change to "Samsung" or "Google" to filter by brand
BRAND = "Google"

df_summary[df_summary["brand"] == BRAND]

,brand,model,date,avg,median,min,max,n
20,Google,Google Pixel 7,2026-04-04,141.67,130.00,76.00,219.0,12
21,Google,Google Pixel 7 Pro,2026-04-04,150.30,150.00,142.99,157.9,3
22,Google,Google Pixel 7a,2026-04-04,120.72,115.00,75.00,160.0,11
23,Google,Google Pixel 8,2026-04-04,220.34,220.00,185.60,299.0,11
24,Google,Google Pixel 8 Pro,2026-04-04,268.01,279.00,160.00,333.0,15
25,Google,Google Pixel 8a,2026-04-04,200.22,199.00,180.00,225.0,6
26,Google,Google Pixel 9,2026-04-04,355.54,304.44,245.40,597.0,8
27,Google,Google Pixel 9 Pro,2026-04-04,420.27,452.36,250.00,519.0,5
28,Google,Google Pixel 9 Pro XL,2026-04-04,492.33,473.00,449.00,555.0,3


## Avg price bar chart — by brand

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

for brand, group in df_summary.groupby("brand"):
    fig, ax = plt.subplots(figsize=(12, 4))
    labels = [m.replace(f"{brand} ", "") for m in group["model"]]
    x      = range(len(labels))

    # min–max range bars
    ax.bar(x, group["max"] - group["min"], bottom=group["min"],
           color="#d0e8ff", edgecolor="#aac8ef", label="Min–Max range")
    # average marker
    ax.scatter(x, group["avg"], color="#1a73e8", zorder=3, label="Avg price", s=40)
    # median marker
    ax.scatter(x, group["median"], color="#e8710a", zorder=3, label="Median price",
               marker="D", s=30)

    ax.set_title(f"{brand} — avg sold price (€)", fontsize=13, fontweight="bold")
    ax.set_xticks(list(x))
    ax.set_xticklabels(labels, rotation=35, ha="right", fontsize=8)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("€%.0f"))
    ax.legend(fontsize=8)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()

## Raw sold listings — look up a specific model

In [ ]:
# Change this to any model name you want to inspect
MODEL = "iPhone 14 Pro"

df_listings = pd.read_sql("""
    SELECT
        sl.title,
        sl.price,
        sl.condition,
        sl.end_time,
        sl.listing_url
    FROM sold_listings sl
    JOIN phone_models pm ON pm.id = sl.phone_model_id
    WHERE pm.model = ?
    ORDER BY sl.end_time DESC
""", con, params=(MODEL,))

print(f"{len(df_listings)} listings found for '{MODEL}'")
df_listings.head(20)

## Price distribution — histogram for a single model

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df_listings["price"], bins=30, color="#1a73e8", edgecolor="white", alpha=0.85)

avg    = df_listings["price"].mean()
median = df_listings["price"].median()
ax.axvline(avg,    color="#e8710a", linestyle="--", linewidth=1.5, label=f"Avg €{avg:.0f}")
ax.axvline(median, color="#34a853", linestyle="--", linewidth=1.5, label=f"Median €{median:.0f}")

ax.set_title(f"Sold price distribution — {MODEL}", fontsize=12, fontweight="bold")
ax.set_xlabel("Price (€)")
ax.set_ylabel("Number of listings")
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()